In [0]:
# Databricks notebook source
# src/pipelines/bronze_pipeline.py

import dlt
from pyspark.sql import functions as F

VOLUME_BASE = "/Volumes/meridian_t02/landing/raw_files"

def add_lineage_metadata(df):
    """Appends mandatory lineage columns to Bronze tables."""
    return (
        df.withColumn("_source_file", F.col("_metadata.file_path"))
          .withColumn("_ingest_time", F.current_timestamp())
          .withColumn("_batch_id", F.regexp_extract(F.col("_metadata.file_path"), r"batch_(\d+)", 1))
    )

# -------------------------------------------------------------------------
# 1. BRONZE CLICKSTREAM (Streaming Table)
# -------------------------------------------------------------------------
@dlt.table(
    name="bronze.bronze_clickstream",
    comment="Append-only raw clickstream events ingested via Auto Loader",
    table_properties={"quality": "bronze"}
)
def bronze_clickstream():
    return (
        spark.readStream.format("cloudFiles")
             .option("cloudFiles.format", "json")
             .option("cloudFiles.inferColumnTypes", "true")
             .load(f"{VOLUME_BASE}/clickstream/*/*")
             .transform(add_lineage_metadata)
    )

# -------------------------------------------------------------------------
# 2. BRONZE REVIEWS (Streaming Table)
# -------------------------------------------------------------------------
@dlt.table(
    name="bronze.bronze_reviews",
    comment="Append-only raw reviews data ingested via Auto Loader",
    table_properties={"quality": "bronze"}
)
def bronze_reviews():
    return (
        spark.readStream.format("cloudFiles")
             .option("cloudFiles.format", "json")
             .option("cloudFiles.inferColumnTypes", "true")
             .load(f"{VOLUME_BASE}/reviews/*/*")
             .transform(add_lineage_metadata)
    )

# -------------------------------------------------------------------------
# 3. BRONZE BOOKINGS (Streaming Table)
# -------------------------------------------------------------------------
@dlt.table(
    name="bronze.bronze_bookings",
    comment="Append-only raw bookings conversion records ingested via Auto Loader",
    table_properties={"quality": "bronze"}
)
def bronze_bookings():
    return (
        spark.readStream.format("cloudFiles")
             .option("cloudFiles.format", "json")
             .option("cloudFiles.inferColumnTypes", "true")
             .load(f"{VOLUME_BASE}/bookings/*/*")
             .transform(add_lineage_metadata)
    )

# -------------------------------------------------------------------------
# 4. BRONZE USERS (Streaming Table)
# -------------------------------------------------------------------------
@dlt.table(
    name="bronze.bronze_users",
    comment="Append-only raw user snapshots and profile changes",
    table_properties={"quality": "bronze"}
)
def bronze_users():
    return (
        spark.readStream.format("cloudFiles")
             .option("cloudFiles.format", "json")
             .option("cloudFiles.inferColumnTypes", "true")
             .load(f"{VOLUME_BASE}/users/*/*")
             .transform(add_lineage_metadata)
    )

# -------------------------------------------------------------------------
# 5. BRONZE PROPERTIES (Streaming Table)
# -------------------------------------------------------------------------
@dlt.table(
    name="bronze.bronze_properties",
    comment="Append-only raw property listing records",
    table_properties={"quality": "bronze"}
)
def bronze_properties():
    return (
        spark.readStream.format("cloudFiles")
             .option("cloudFiles.format", "json")
             .option("cloudFiles.inferColumnTypes", "true")
             .load(f"{VOLUME_BASE}/properties/*/*")
             .transform(add_lineage_metadata)
    )